In [ ]:
# Import necessary libraries
import nltk
nltk.download('punkt')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import torch

# For BARTScore and GPTScore
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# For QSTS and QRelScore
from sentence_transformers import SentenceTransformer, util

# Define the context sentence and generated MCQ question
# context_sentence = "Machine learning is a field of artificial intelligence that uses algorithms to learn from and make predictions on data."

# generated_mcq_question = "What is machine learning?"
# options = [
#     "A field of artificial intelligence that uses algorithms",
#     "A form of supervised learning only",
#     "A programming language",
#     "A method of organizing data"
# ]
context_sentence = "A high-degree polynomial kernel introduces high complexity to the decision boundary, which can lead to overfitting, especially when the training dataset is small. The SVM model may fit the noise in the training data rather than capturing the underlying pattern, reducing generalization to unseen data."
generated_mcq_question = "In the context of Support Vector Machines (SVM), which of the following scenarios is most likely to lead to overfitting when classifying a dataset?"
options = [
    "Choosing a linear kernel for a dataset that is linearly separable.",
    "Using a high-degree polynomial kernel on a small training dataset.",
    "Selecting a Gaussian (RBF) kernel with a large value for the hyperparameter 𝛾"
    "Setting the regularization parameter C to a very small value."
]

mcq_text = generated_mcq_question + " " + " ".join(options)

In [ ]:
# Compute BLEU-4 score

# BLEU-4 Calculation
reference_tokens = [context_sentence.split()]
generated_tokens = mcq_text.split()

smooth_fn = SmoothingFunction().method4
bleu_score = sentence_bleu(reference_tokens, generated_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)

# Display result
print(f"BLEU-4 score: {bleu_score:.4f}")

In [ ]:
# Compute ROUGE-L score
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = scorer.score(context_sentence, mcq_text)
rouge_l = scores['rougeL'].fmeasure

print(f"ROUGE-L Score: {rouge_l:.4f}")

In [ ]:
# Compute BERTScore
P, R, F1 = bert_score_func(
    [generated_mcq_question],
    [context_sentence],
    lang="en",
    verbose=True
)
print(f"BERTScore - Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

In [ ]:
# Compute BARTScore (reference-based)
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

def bart_score(reference, hypothesis):
    inputs = tokenizer(reference, hypothesis, return_tensors='pt', truncation=True)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        log_likelihood = outputs.loss.item()
    return -log_likelihood  # Lower loss means higher score

score_bart_ref = bart_score(context_sentence, mcq_text)

print(f"BARTScore-ref: {score_bart_ref:.4f}")

In [ ]:
# Compute GPTScore (reference-based)
model_gpt = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer_gpt = GPT2Tokenizer.from_pretrained('gpt2')

def gpt_score(reference, hypothesis):
    text = reference + " " + hypothesis
    inputs = tokenizer_gpt(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model_gpt(**inputs, labels=inputs["input_ids"])
        log_likelihood = outputs.loss.item()
    return -log_likelihood

score_gpt_ref = gpt_score(context_sentence, mcq_text)

print(f"GPTScore-ref: {score_gpt_ref:.4f}")

In [ ]:
# Compute QSTS (Question Semantic Textual Similarity)
model_st = SentenceTransformer('all-MiniLM-L6-v2')

embedding_context = model_st.encode(context_sentence, convert_to_tensor=True)
embedding_question = model_st.encode(generated_mcq_question, convert_to_tensor=True)

similarity = util.pytorch_cos_sim(embedding_context, embedding_question).item()

print(f"QSTS (Semantic Similarity): {similarity:.4f}")

In [ ]:
# Compute BARTScore (source-based)
# Reusing the previously defined bart_score function

score_bart_src = bart_score(context_sentence, generated_mcq_question)

print(f"BARTScore-src: {score_bart_src:.4f}")

In [ ]:
# Compute GPTScore (source-based)
# Reusing the previously defined gpt_score function

score_gpt_src = gpt_score(context_sentence, generated_mcq_question)

print(f"GPTScore-src: {score_gpt_src:.4f}")

In [ ]:
# Compute QRelScore (Question Relevance Score)
# Using semantic similarity as a proxy for relevance

embedding_sent = model_st.encode(context_sentence, convert_to_tensor=True)
embedding_q = model_st.encode(generated_mcq_question, convert_to_tensor=True)

similarity_rel = util.pytorch_cos_sim(embedding_sent, embedding_q).item()

print(f"QRelScore (Relevance): {similarity_rel:.4f}")